# Portfolio-safe version

This notebook is a sanitized portfolio adaptation of the author's MSc Data Analytics project.
Environment-specific paths, cloud bucket names, notebook outputs, and exact patient examples
have been removed or generalized. The original methodology and core code structure are preserved.

**Data note:** the underlying TCIA imaging/clinical data are not redistributed in this repository.
Configure your own authorized/local dataset paths before running the notebook.


# Clinical Survival Analysis - STS (Soft Tissue Sarcoma)

This notebook reads **Sheet1 = Clinical Information** and **Sheet2 = Outcome vector** from
"`INFOclinical_STS.xlsx`",

merges them by **Patient ID** with **Sheet2 overriding non-null values**,
and performs:

- **OS** analysis: time = `Time - diagnosis to last follow-up (days)`; event = 1 for `D`, otherwise 0.
- **PFS** analysis: time = `Time - diagnosis to outcome (days)`; event = 1 for `D` or `AWD`, 0 for `NED`.

It produces Kaplan–Meier plots (overall and by Sex, MSKCC type, Grade), log-rank tests, and a small Cox model.

All figures are **displayed inline** and also saved into folder "`results_step04_clinical_only/`" with a "`04_`" prefix.

In [ ]:
# Step 1: Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
import os

sns.set(style="whitegrid")
RESULTS_DIR = "results/clinical_survival"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Artifacts will be saved to:", RESULTS_DIR)

In [ ]:
# Step 2: Load and MERGE Sheet1 + Sheet2 with Sheet2 overriding non-null
file_path = "data/INFOclinical_STS.xlsx"  # ensure file is in same directory
xls = pd.ExcelFile(file_path)
sheet1 = xls.parse('Clinical Information')
sheet2 = xls.parse('Outcome vector')

# Normalize Patient ID name if needed
if 'Patient ID' not in sheet1.columns:
    raise ValueError("'Patient ID' not found in Clinical Information sheet")
if 'Patient ID' not in sheet2.columns:
    raise ValueError("'Patient ID' not found in Outcome vector sheet")

# First, set index to Patient ID
s1 = sheet1.set_index('Patient ID')
s2 = sheet2.set_index('Patient ID')

# Use combine_first so Sheet2 (s2) overrides where it has non-null values
merged = s2.combine_first(s1).reset_index()
print("Merged shape:", merged.shape)
merged.to_csv(f"{RESULTS_DIR}/04_merge_snapshot.csv", index=False)

# Replace the common placeholder
merged = merged.replace("--", np.nan)

In [ ]:
# Step 3: Prepare columns (ensure numeric for time) and define OS & PFS frames
def to_num(s):
    return pd.to_numeric(s, errors='coerce')

col_time_outcome = 'Time – diagnosis to outcome (days)'
col_time_lastfu = 'Time – diagnosis to last follow-up (days)'
col_status = 'Status (NED, AWD, D)'

for c in [col_time_outcome, col_time_lastfu]:
    if c in merged.columns:
        merged[c] = to_num(merged[c])

def map_event(status, mode='OS'):
    if pd.isna(status):
        return np.nan
    s = str(status).strip().upper()
    if mode == 'OS':
        return 1 if s == 'D' else (0 if s in ['NED', 'AWD'] else np.nan)
    else:  # PFS
        return 1 if s in ['D', 'AWD'] else (0 if s == 'NED' else np.nan)

# OS frame
os_df = merged[[
    'Patient ID', 'Age', 'Sex', 'Grade', 'MSKCC type',
    col_time_lastfu, col_status
]].copy()
os_df.rename(columns={col_time_lastfu: 'Time', col_status: 'Status'}, inplace=True)
os_df['Event'] = os_df['Status'].apply(lambda x: map_event(x, mode='OS'))
os_df = os_df.dropna(subset=['Time', 'Event'])
os_df = os_df[os_df['Time'] >= 0]
print("OS N:", len(os_df))
os_df.to_csv(f"{RESULTS_DIR}/04_os_frame.csv", index=False)

# PFS frame
pfs_df = merged[[
    'Patient ID', 'Age', 'Sex', 'Grade', 'MSKCC type',
    col_time_outcome, col_status
]].copy()
pfs_df.rename(columns={col_time_outcome: 'Time', col_status: 'Status'}, inplace=True)
pfs_df['Event'] = pfs_df['Status'].apply(lambda x: map_event(x, mode='PFS'))
pfs_df = pfs_df.dropna(subset=['Time', 'Event'])
pfs_df = pfs_df[pfs_df['Time'] >= 0]
print("PFS N:", len(pfs_df))
pfs_df.to_csv(f"{RESULTS_DIR}/04_pfs_frame.csv", index=False)

In [ ]:
# Step 4: KM helpers
def plot_km(df, time_col, event_col, title, out_png):
    kmf = KaplanMeierFitter()
    plt.figure(figsize=(8, 6))
    kmf.fit(df[time_col], df[event_col], label=title)
    kmf.plot(ci_show=True)
    plt.title(title)
    plt.xlabel("Time (days)")
    plt.ylabel("Survival Probability")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.show()

def plot_km_logrank(df, by_col, time_col, event_col, mode_tag, out_prefix):
    if by_col not in df.columns:
        print(f"[WARN] Column not found for KM stratification: {by_col}")
        return
    groups = df[by_col].dropna().unique()
    if len(groups) < 2:
        print(f"[WARN] Not enough groups in {by_col} to plot KM strata.")
        return
    plt.figure(figsize=(8, 6))
    kmf = KaplanMeierFitter()
    for group in groups:
        mask = df[by_col] == group
        kmf.fit(df[mask][time_col], df[mask][event_col], label=str(group))
        kmf.plot(ci_show=True)
    plt.title(f"Kaplan-Meier — {mode_tag} by {by_col}")
    plt.xlabel("Time (days)")
    plt.ylabel("Survival Probability")
    plt.grid(True)
    plt.tight_layout()
    out_png = f"{RESULTS_DIR}/{out_prefix}_{mode_tag}_by_{by_col.replace(' ', '_')}.png"
    plt.savefig(out_png, dpi=300)
    plt.show()

    # Log-rank test
    if len(groups) == 2:
        g1 = df[df[by_col] == groups[0]]
        g2 = df[df[by_col] == groups[1]]
        results = logrank_test(g1[time_col], g2[time_col], g1[event_col], g2[event_col])
        print(f"Log-rank test ({groups[0]} vs {groups[1]}): p = {results.p_value:.4f}")
    elif len(groups) > 2:
        results = multivariate_logrank_test(df[time_col], df[by_col], df[event_col])
        print(f"Multivariate log-rank test ({by_col}): p = {results.p_value:.4f}")

In [ ]:
# Step 5: KM – OS and PFS overall + by Sex, MSKCC type, Grade
if len(os_df):
    plot_km(os_df, 'Time', 'Event', 'Kaplan-Meier – Overall Survival (OS)', f"{RESULTS_DIR}/04_os_km_overall.png")
    for col in ['Sex', 'MSKCC type', 'Grade']:
        plot_km_logrank(os_df, col, 'Time', 'Event', 'OS', '04_os_km')

if len(pfs_df):
    plot_km(pfs_df, 'Time', 'Event', 'Kaplan-Meier – Progression-Free Survival (PFS)', f"{RESULTS_DIR}/04_pfs_km_overall.png")
    for col in ['Sex', 'MSKCC type', 'Grade']:
        plot_km_logrank(pfs_df, col, 'Time', 'Event', 'PFS', '04_pfs_km')

In [ ]:
# Step 6: Small Cox model (Age, Sex, Grade) for OS and PFS
def run_cox_small(df, mode_tag):
    use_cols = [c for c in ['Age','Sex','Grade'] if c in df.columns]
    if not use_cols:
        print(f"[{mode_tag}] No covariates available for Cox model.")
        return
    cox_df = df[['Time','Event'] + use_cols].copy()
    # Encode categoricals
    for c in ['Sex','Grade']:
        if c in cox_df.columns:
            cox_df[c] = cox_df[c].astype('category')
    cox_df = pd.get_dummies(cox_df, drop_first=True)
    cph = CoxPHFitter(penalizer=0.25)
    cph.fit(cox_df, duration_col='Time', event_col='Event')
    cph.print_summary()
    # Save summary
    cph.summary.to_csv(f"{RESULTS_DIR}/04_{mode_tag}_cox_summary.csv")
    with open(f"{RESULTS_DIR}/04_{mode_tag}_cox_cindex.txt", 'w') as f:
        f.write(f"C-index: {float(cph.concordance_index_):.4f}\n")
    # Plot HRs
    ax = cph.plot()
    plt.title(f"Cox Model – {mode_tag}")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/04_{mode_tag}_cox_plot.png", dpi=300)
    plt.show()

if len(os_df) >= 5:
    run_cox_small(os_df, 'os')
else:
    print("[OS] Too few rows for stable Cox model; skipping.")

if len(pfs_df) >= 5:
    run_cox_small(pfs_df, 'pfs')
else:
    print("[PFS] Too few rows for stable Cox model; skipping.")

In [ ]:
# Step 7: Export key tables for the report
os_df.to_csv(f"{RESULTS_DIR}/04_os_analysis_frame.csv", index=False)
pfs_df.to_csv(f"{RESULTS_DIR}/04_pfs_analysis_frame.csv", index=False)
print("Saved analysis frames and figures in:", RESULTS_DIR)